#imports


In [ ]:
!pip install pingouin
!pip install qlatent
%pip install --quiet git+https://github.com/cnai-lab/qpsychometric.git


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.4/204.4 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 914.1/914.1 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 47.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.2.3 which is incompatible.


In [ ]:
import torch
import pandas as pd
from pathlib import Path
import gc
from tqdm.auto import tqdm
import warnings
import pingouin as pg
from sentence_transformers import SentenceTransformer, util
from pathlib import Path
import json, torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import DataLoader
from tqdm import tqdm
import numpy as np
from qlatent.qmnli.qmnli import *
from qlatent.qmnli.qmnli import _QMNLI, QMNLI


device = 0 if torch.cuda.is_available() else -1
print(device)

0


In [ ]:
softmax_files = [True, False]

def split_question(Q, index, scales, softmax, filters):
  result = []
  for s in scales:
    q = QCACHE(Q())
    for sf in softmax:
      for f in filters:
        if sf:
            qsf = QSOFTMAX(q,dim=[index[0], s])
            qsf_f = QFILTER(qsf,filters[f],filtername=f)
            print((index, s),sf,f)
            result.append(qsf_f)

            qsf = QSOFTMAX(q,dim=s)
            qsf_f = QFILTER(qsf,filters[f],filtername=f)
            print(s,sf,f)
            result.append(qsf_f)

            qsf = QSOFTMAX(q,dim=index[0])
            qsf_f = QFILTER(qsf,filters[f],filtername=f)
            print(index[0],sf,f)
            result.append(qsf_f)
        else:
            qsf = QPASS(q,descupdate={'softmax':''})
            qsf_f = QFILTER(qsf,filters[f],filtername=f)
            print(s,sf,f)
            result.append(qsf_f)
  return result

#load models


In [ ]:
p = 'valhalla/distilbart-mnli-12-6'
mnli = pipeline("zero-shot-classification",device=device, model=p)
mnli.model_identifier = p

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.23G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.23G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

Device set to use cuda:0


In [ ]:
gc.collect()
torch.cuda.empty_cache()

123

#linguistic acceptability


In [ ]:
sentence_embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
cola = pipeline("text-classification","mrm8488/deberta-v3-small-finetuned-cola", device=device)

import os
import pandas as pd
from nltk.translate.bleu_score import sentence_bleu

def linguistic_acceptabilities(q, index, scale, question_name, student_id, output_path=Path(''), save_to_file=False):
    score_by_cola_lst=[]
    score_of_semantic_distance_lst=[]
    score_by_bleu_lst=[]
    kmap_lst=[]
    question_name_lst=[]
    description = q._descriptor
    strFactor=description['Factor']
    strOrdinal=str(description.get('Ordinal', 0))
    ##cleaning the string to get the original question
    strOriginal= description['Original']
    strOriginal = 'none' if strOriginal is None else strOriginal
    strOriginal=strOriginal.replace(strFactor,'',1)
    strOriginal=strOriginal.replace(strOrdinal,'',1)
    strOriginal=strOriginal.replace('.','',1)
    strOriginal=strOriginal.strip() #the original question
    rows = []

    partial_internal_consistency = partial(q.internal_consistency, filter={}, index=index , scale=scale)
    try:
        silhouette_score = partial_internal_consistency(measure='silhouette_score', metric='correlation')
    except Exception as e:
        print(e)
        print('silhouette_score is set to -1')
        silhouette_score = -1

    if hasattr(q, 'linguistic_acceptability'):
        q.linguistic_acceptability['silhouette_score'] = silhouette_score
        return q.linguistic_acceptability

    for kmap in q._keywords_map:
        score = {}
        score['question_name'] = question_name
        context = q._context_template.format_map(kmap)
        answer = q._answer_template.format_map(kmap)
        score['original_question'] = strOriginal


        cola_score = cola(context +" "+ answer)[0].get('score')
        score['cola_score'] = cola_score
        score['param'] = kmap
        strPermutation= context +" "+ answer
        # sentences = [context +" "+ answer]
        score['question_permutation'] = strPermutation
        #Compute embedding for both lists
        embeddings1 = sentence_embedding_model.encode(strOriginal, convert_to_tensor=True)
        embeddings2 = sentence_embedding_model.encode(strPermutation, convert_to_tensor=True)

        #Compute cosine-similarities
        cosine_scores = util.cos_sim(embeddings1, embeddings2)
        score['semantic_similarity'] = cosine_scores.item()

        score['silhouette_score'] = silhouette_score
        rows.append(score)


    filename = output_path / 'linguistic_acceptabilities.csv'
    df = pd.DataFrame(rows)
    df['student_id'] = student_id
    df = df[['student_id', 'question_name','original_question', 'param','question_permutation','cola_score','semantic_similarity','silhouette_score']]
    if save_to_file:
        if filename.exists():
            df.to_csv(filename, index=False, header=None, mode='a', encoding='utf-8-sig')
        else:
            df.to_csv(filename, index=False, encoding='utf-8-sig')
#     print(f"Linguistic acceptabilities saved in {filename}")
    q.linguistic_acceptability = df
    return df

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/393 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/18.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(
Device set to use cuda:0


#load questionaire


In [ ]:
from qpsychometric.mental_health.patient_health_questionnaire import phq_questionnaire

# Load PHQ-9 (QMNLI task) and unpack items
phq9_qmnli_df = phq_questionnaire['QMNLI']
phq9_questions = phq9_qmnli_df.get_questions()   # list of 9 question *classes*
assert len(phq9_questions) == 9, f"Expected 9 items, got {len(phq9_questions)}"

PHQ9Q1, PHQ9Q2, PHQ9Q3, PHQ9Q4, PHQ9Q5, PHQ9Q6, PHQ9Q7, PHQ9Q8, PHQ9Q9 = phq9_questions

# Build splits per item (uses your existing split_question + softmax_files)
Q1s = split_question(PHQ9Q1, index=["index"], scales=["frequency"], softmax=softmax_files,
                     filters={'unfiltered': {}, "positiveonly": PHQ9Q1().get_filter_for_postive_keywords(['frequency'])})
Q2s = split_question(PHQ9Q2, index=["index"], scales=["frequency"], softmax=softmax_files,
                     filters={'unfiltered': {}, "positiveonly": PHQ9Q2().get_filter_for_postive_keywords(['frequency'])})
Q3s = split_question(PHQ9Q3, index=["index"], scales=["frequency"], softmax=softmax_files,
                     filters={'unfiltered': {}, "positiveonly": PHQ9Q3().get_filter_for_postive_keywords(['frequency'])})
Q4s = split_question(PHQ9Q4, index=["index"], scales=["frequency"], softmax=softmax_files,
                     filters={'unfiltered': {}, "positiveonly": PHQ9Q4().get_filter_for_postive_keywords(['frequency'])})
Q5s = split_question(PHQ9Q5, index=["index"], scales=["frequency"], softmax=softmax_files,
                     filters={'unfiltered': {}, "positiveonly": PHQ9Q5().get_filter_for_postive_keywords(['frequency'])})
Q6s = split_question(PHQ9Q6, index=["index"], scales=["frequency"], softmax=softmax_files,
                     filters={'unfiltered': {}, "positiveonly": PHQ9Q6().get_filter_for_postive_keywords(['frequency'])})
Q7s = split_question(PHQ9Q7, index=["index"], scales=["frequency"], softmax=softmax_files,
                     filters={'unfiltered': {}, "positiveonly": PHQ9Q7().get_filter_for_postive_keywords(['frequency'])})
Q8s = split_question(PHQ9Q8, index=["index"], scales=["frequency"], softmax=softmax_files,
                     filters={'unfiltered': {}, "positiveonly": PHQ9Q8().get_filter_for_postive_keywords(['frequency'])})
Q9s = split_question(PHQ9Q9, index=["index"], scales=["frequency"], softmax=softmax_files,
                     filters={'unfiltered': {}, "positiveonly": PHQ9Q9().get_filter_for_postive_keywords(['frequency'])})

# Pick the first variant (e.g., joint-softmax + unfiltered) for each
q1 = Q1s[0]; q2 = Q2s[0]; q3 = Q3s[0]; q4 = Q4s[0]; q5 = Q5s[0]
q6 = Q6s[0]; q7 = Q7s[0]; q8 = Q8s[0]; q9 = Q9s[0]


(['index'], 'frequency') True unfiltered
frequency True unfiltered
index True unfiltered
(['index'], 'frequency') True positiveonly
frequency True positiveonly
index True positiveonly
frequency False unfiltered
frequency False positiveonly
(['index'], 'frequency') True unfiltered
frequency True unfiltered
index True unfiltered
(['index'], 'frequency') True positiveonly
frequency True positiveonly
index True positiveonly
frequency False unfiltered
frequency False positiveonly
(['index'], 'frequency') True unfiltered
frequency True unfiltered
index True unfiltered
(['index'], 'frequency') True positiveonly
frequency True positiveonly
index True positiveonly
frequency False unfiltered
frequency False positiveonly
(['index'], 'frequency') True unfiltered
frequency True unfiltered
index True unfiltered
(['index'], 'frequency') True positiveonly
frequency True positiveonly
index True positiveonly
frequency False unfiltered
frequency False positiveonly
(['index'], 'frequency') True unfiltered

# Run Questionnaires on models

## Utility functions

In [ ]:
def question_attributes(q):
    score = {}
    score['questionnair']=q._descriptor['Questionnair']
    score['factor']=q._descriptor['Factor']
    score['ordinal']=q._descriptor['Ordinal']
    score['scale']=q._descriptor['scale']
    score['index']=q._descriptor['index']
    score['filter']=q._descriptor['filter']
    score['softmax'] = q._descriptor['softmax']
    score["original"] = q._descriptor['Original']
    score['Q'] = f"{score['questionnair']}{score['factor']}{score['ordinal']}"
    score['context_template'] = q._context_template
    score['answer_template'] = q._answer_template
    score['dimensions'] = q._dimensions
    score['model'] = q.model.model_identifier if q.model else ""
    return score

def get_question_features(q, student_id='student_id', output_path=Path(''), save_to_file=False):
    score = question_attributes(q)
    score['mean_score'] = q.mean_score()
    index= q._index
    scale= q._scale
    linguistic_df = linguistic_acceptabilities(q, index=index, scale=scale,question_name=score['Q'], student_id=student_id,
                                               output_path=output_path, save_to_file=save_to_file)
    row = linguistic_df[['cola_score','silhouette_score']].mean(axis=0)
    row_dict = dict(row)
    row_dict['semantic_similarity'] = linguistic_df['semantic_similarity'].quantile(0.75)
    score = score | row_dict
    return score

def extract_epoch(model_path):
    if 'epoch-' in model_path.name:
        i = model_path.name.find('epoch-')
        j = model_path.name.find('_', i)
        if j > 0:
            epoch = int(model_path.name[i+len('epoch-'):j])
        else:
            epoch = int(model_path.name[i+len('epoch-'):])

    elif 'checkpoint-' in model_path.name:
        i = model_path.name.find('checkpoint-')
        j = model_path.name.find('_', i)
        if j > 0:
            epoch = int(model_path.name[i+len('checkpoint-'):j])
        else:
            epoch = int(model_path.name[i+len('checkpoint-'):])
    else:
        epoch = 0
    return epoch

def extract_run(model_path):
    try:
        if 'run' in model_path.name:
            for part in model_path.name.split('_'):
                if 'run' in part:
                    return int(part.replace('run', ''))
        else:
            return -1
    except Exception as e:
        print(e)
        return -1

import json

def get_mnli_score(checkpoint_path):
    mnli_score_path = checkpoint_path / 'all_results.json'
    if not mnli_score_path.exists():
        mnli_score_path = checkpoint_path.parent / (checkpoint_path.name + '_mnli_eval') / 'all_results.json'
    if mnli_score_path.exists():
        with open(mnli_score_path) as f:
            return json.load(f)["eval_accuracy"]
    else:
        return -1


def run_questions(questions, mnli_checkpoint, train_process, fintune_dataset, q_range=[5, 0]):
    rows = []
    checkpoint = Path(mnli_checkpoint.model_identifier)
    for q_raw in tqdm(questions):
        T = time.time()
        q = q_raw.run(mnli_checkpoint)
        T = time.time()
        score = get_question_features(q)
        score['epoch'] = extract_epoch(checkpoint)
        score['train_process'] = train_process
        score['dataset'] = fintune_dataset
        score['run'] = extract_run(checkpoint.parent)
        score['mnli_score'] = get_mnli_score(checkpoint)
        score['range'] = (q._weights_flat.min(), q._weights_flat.max())
        score['ASI_score'] = np.interp(score['mean_score'], [q._weights_flat.min(), q._weights_flat.max()], q_range)
        rows.append(score)
        gc.collect()
        torch.cuda.empty_cache()
    return rows


def calc_scores(questions, checkpoint, output_path, train_process, fintune_dataset, q_range=[5, 0]):
    fix_config(checkpoint)
    mnli_checkpoint = pipeline("zero-shot-classification", str(checkpoint), device=device)
    mnli_checkpoint.model_identifier = str(checkpoint)
    rows = run_questions(questions, mnli_checkpoint, train_process, fintune_dataset=fintune_dataset, q_range=q_range)
    return rows

def add_epochs_to_rows(rows, mlm_epoch, mnli_checkpoint):
    for score in rows:
        score['mlm_epoch'] = mlm_epoch
        score['mnli_checkpoint'] = mnli_checkpoint
    return rows


def write_to_csv(rows, output_path):
    old_score_hostile_df = pd.DataFrame(rows)
    if output_path.exists():
        old_score_hostile_df.to_csv(output_path, index=False, header=None, mode='a')
    else:
        old_score_hostile_df.to_csv(output_path, index=False)

def fix_config(checkpoint):
    if checkpoint.exists():
        with open(checkpoint / 'config.json') as f:
            d1 = json.load(f)
        d1['id2label'] = {'0': 'entailment', '1': 'neutral', '2': 'contradiction'}
        d1['label2id'] = {'contradiction': 2, 'entailment': 0, 'neutral': 1}
        with open(checkpoint / 'config.json', 'w') as f:
            json.dump(d1, f)
    else:
        print(checkpoint, '#### Not exists ####')

def calc_for_all_models(Qs, q_range= [5, 0]):
    all_rows = []
    for p in tqdm(mnli_pipelines):
        print(p)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            rows = calc_scores(Qs, Path(p),  Path(p), '->'.join(['base']), 'hostile',
                               use_base_model=False, q_range=q_range)
            rows = add_epochs_to_rows(rows, 0, 0)
            all_rows += rows
    return pd.DataFrame(all_rows)

## Run Questions

In [ ]:
result_path = Path('results/')
if not result_path.exists():
    os.makedirs(result_path)

In [ ]:
mnli_pipelines = [
    'typeform/distilbert-base-uncased-mnli',
    'typeform/mobilebert-uncased-mnli',
    'cross-encoder/nli-roberta-base',
    'cross-encoder/nli-deberta-base',
    'cross-encoder/nli-distilroberta-base',
    'cross-encoder/nli-MiniLM2-L6-H768',
    'navteca/bart-large-mnli',
    'digitalepidemiologylab/covid-twitter-bert-v2-mnli',
    'joeddav/bart-large-mnli-yahoo-answers',
    'Narsil/deberta-large-mnli-zero-cls',
    'microsoft/deberta-large-mnli',
    'microsoft/deberta-base-mnli',
    'Alireza1044/albert-base-v2-mnli',
    'yoshitomo-matsubara/bert-large-uncased-mnli',
    'yoshitomo-matsubara/bert-base-uncased-mnli',
    'yoshitomo-matsubara/bert-base-uncased-mnli_from_bert-large-uncased-mnli',
    'valhalla/distilbart-mnli-12-6',
]


In [ ]:
from collections import defaultdict

questions = Q2s + Q4s + Q5s + Q7s
questions += Q1s + Q6s + Q3s + Q8s + Q9s

update = True

output_path = result_path / f'phq9_mnli_check1.csv'
pipelines = mnli_pipelines

if output_path.exists():
    temp_df = pd.read_csv(output_path)
    indexes = temp_df.groupby(['model', 'Q']).count().index.values
    used_models = defaultdict(set)
    for k, v in indexes:
        used_models[k].add(v)
else:
    used_models = {}


for p in tqdm(pipelines):
    if get_mnli_score(Path(p)) < 0.7 and p not in mnli_pipelines:
        print('Skip:', p)
        continue
    with warnings.catch_warnings():
        try:
            warnings.simplefilter("ignore")
            if p in used_models and not update:
                pipline_questions = []
                for q in questions:
                    if question_attributes(q)['Q'] not in used_models[p]:
                        pipline_questions.append(q)
                    else:
                        print('skip', p, question_attributes(q)['Q'])
            else:
                pipline_questions = questions

            rows = calc_scores(pipline_questions, Path(p),  output_path, '->'.join(['base']), 'hostile',)
            rows = add_epochs_to_rows(rows, 0, 0)
            write_to_csv(rows, output_path)
            gc.collect()
            torch.cuda.empty_cache()
        except Exception as e:
            print(e)


df = pd.read_csv(output_path)
df = df.drop_duplicates(subset=['filter','softmax','model','Q'], keep='last')
df.to_csv(output_path, index=False)

  0%|          | 0/17 [00:00<?, ?it/s]

typeform/distilbert-base-uncased-mnli #### Not exists ####


config.json:   0%|          | 0.00/776 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/258 [00:00<?, ?B/s]

...


 99%|█████████▊| 71/72 [00:33<00:00,  1.94it/s]

Number of labels is 1. Valid values are 2 to n_samples - 1 (inclusive)
silhouette_score is set to -1



100%|██████████| 72/72 [00:34<00:00,  2.10it/s]


0

100%|██████████| 17/17 [16:48<00:00, 59.31s/it]


# Validations

In [ ]:
def load_results(csv_path, softmax, positiveonly, value='phq9_score', index='model'):
    df = pd.read_csv(csv_path)
    df['model'] = df['model'].str.replace('/dt/puzis/cnalab/maor/', '')
    if df['softmax'].isna().sum() > 0:
        softmax_filter = df['softmax'].isna()
    else:
        softmax_filter = df['softmax'] == ''
    if softmax:
        df = df[df['softmax'] == str(softmax)]
    else:
        df = df[softmax_filter]
    if value != 'silhouette_score':
        pass
    else:
        df = df[df['silhouette_score'] > -1]
    if positiveonly:
        df = df[df['filter']=="positiveonly"]
    else:
        df = df[df['filter']=="unfiltered"]
    results_df = pd.pivot_table(df, values=value, index=index, columns='Q', aggfunc='mean')
    return results_df

In [ ]:
softmax_phq = []   # match whatever you logged in the CSV
positiveonly = True                    # or False, depending on which slice you want

# path to the results CSV for GAD-7 (adjust filename as needed)
q_path = result_path / 'phq9_mnli_check1.csv'


## Semantic Validation

In [ ]:

cols = ['semantic_similarity', 'cola_score', 'silhouette_score']

results = []
for softmax_filter in [softmax_phq]:
    q_res = [
        load_results(q_path, softmax=softmax_filter, positiveonly=False, value=v).mean(axis=0)
        for v in cols
    ]
    results.append(pd.concat(q_res, axis=1))

linguistic_acceptability_df = pd.concat(results, axis=0)
linguistic_acceptability_df.columns = ['semantic_similarity', 'cola_score', 'silhouette_score']

# save + show
linguistic_acceptability_df.to_csv(result_path / 'phq9_linguistic_acceptability.csv', index=False)
display(linguistic_acceptability_df)

print('Semantic means:\n', linguistic_acceptability_df.mean())
print('Semantic stds:\n', linguistic_acceptability_df.std())


,semantic_similarity,cola_score,silhouette_score
Q,,,
PHQ9PHQ91,0.614383,0.830769,0.742765
PHQ9PHQ92,0.580788,0.927962,0.647594
PHQ9PHQ93,0.631983,0.924723,0.349647
PHQ9PHQ94,0.644818,0.895164,0.740984
PHQ9PHQ95,0.671451,0.680076,0.672359
PHQ9PHQ96,0.508153,0.891674,0.680217
PHQ9PHQ97,0.667652,0.826986,0.613143
PHQ9PHQ98,0.686993,0.796706,0.643557
PHQ9PHQ99,0.485451,0.773650,0.374603


Semantic means:
 semantic_similarity    0.610186
cola_score             0.838634
silhouette_score       0.607208
dtype: float64
Semantic stds:
 semantic_similarity    0.072008
cola_score             0.081178
silhouette_score       0.145463
dtype: float64


## Internal Consistency

In [ ]:
def get_factor_sub_features(factor, data_df):
    feature_subset = []
    for subset in factor:
        feature_subset += [c for c in data_df.columns if subset in c]
    return list(set(feature_subset))

In [ ]:
value='mean_score'

results = []
for softmax_filter in [softmax_phq]:
    results.append(load_results(q_path,softmax=softmax_filter,positiveonly=positiveonly, value=value))

data_df = pd.concat(results, axis=1)

print('Cronbach Alpha:')


alpha = pg.cronbach_alpha(data=data_df)
print(f'phq9, Alpha:, {alpha}')

Cronbach Alpha:
phq9, Alpha:, (np.float64(0.9029077275716181), array([0.798, 0.965]))
